# Ranking predictions — models and saved results

**Question (a)**: the day before a season starts, predict the final regular-season ranking of every
team *inside its conference* (two rankings of ~15 teams per season, ordered by points).

This notebook is the **only place where models are trained**. It uses a deliberately small set:

| Family | Model | Why it is kept |
|---|---|---|
| Naive | Random ranking | floor with no skill (scale of the metrics) |
| Naive | Previous available ranking | strongest simple history baseline |
| External | Ridge + external | regularised linear model on the selected features |
| External | Gradient boosting + external | non-linear model on the same features |
| External | Ensemble + external | average of the two models above (more stable) |
| Pairwise | Pairwise logistic | simple learning-to-rank formulation |
| Pairwise | Pairwise boosting | non-linear pairwise ranking |

So: **2 naive baselines + 5 models**. The other models tested during the project
(random forest, pairwise blend, base team-history models) are kept in `archive/` for traceability.

Everything is evaluated with the same walk-forward protocol:

> for each test season *Y*, train on every season `< Y` and predict *Y*.

The target is the **points percentage** of the season (``Pts / (2 * GP)``), expressed as a
league-wide z-score. Points percentage is comparable across 80-game, 82-game and shortened
(48-game) seasons, unlike raw points or wins.

**Leak policy.** No feature uses information from the season being predicted:

* every team-performance feature is lagged (season *Y-1* predicts *Y*);
* coach columns (`coach_changed`, tenure, interim, number of coaches) are excluded — they are the
  target of the separate coach-change task;
* current-season roster composition is excluded — the roster is only known after the season
  (in-season arrivals).

Two splits are stored:

* `clean` — 2005–2009, never used for design decisions;
* `backtest` — 1995–2009 (except the 2004 lockout), a stability/power check. Seasons 1995–2004
  were used for feature selection, so this split is **not** a clean out-of-sample test.

At the end, the notebook writes:

```
results/predictions.csv        one row per (split, model, season, team)
results/metrics_by_season.csv  one row per (split, model, season, conference)
results/summary.csv            one row per (split, model)
results/run_info.json          features, years, model descriptions
```

`ranking_results_analysis.ipynb` only reads these files; it never trains a model.

## 1. Data and features

* Base dataset: `data_prepared/teams.csv`, prepared by `data_preparation.ipynb`.
* External features: built by `ranking_features.py` from the raw tables.
* Selected features: `selected_features.json`, produced by `ranking_feature_analysis.ipynb`.

The external models use the selected leak-free features (history + shots/luck + any other block
kept by the selection). The pairwise models use the base feature set, so that the pairwise
formulation can be compared to the external-feature gain.

In [1]:
import json
import sys
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
    VotingRegressor,
)

# Make the helper modules importable whether the notebook is run from the project root
# or from the ranking/ directory.
for candidate in [Path.cwd(), Path.cwd() / "ranking"]:
    if (candidate / "ranking_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import ranking_utils as ru
import ranking_features as rf

RS = ru.RANDOM_STATE
CLEAN_YEARS = [2005, 2006, 2007, 2008, 2009]
BACKTEST_YEARS = [y for y in range(1995, 2010) if y != 2004]
SPLITS = {"clean": CLEAN_YEARS, "backtest": BACKTEST_YEARS}

# ---- datasets --------------------------------------------------------------
teams = ru.load_teams()
df = ru.build_dataset(teams)

selection = json.loads(ru.SELECTED_FEATURES_PATH.read_text())
BASE_FEATURES = selection["base_features"]
EXT_FEATURES = selection["features"]
SELECTED_GROUPS = selection["selected_groups"]

ext, EXT_GROUPS = rf.build_external_features(ru.DATA_DIR)
df_ext = df.merge(ext.drop(columns=["franchID"]), on=["year", "tmID"], how="left")
EXT_COLS = [c for cols in EXT_GROUPS.values() for c in cols]
df_ext[EXT_COLS] = df_ext[EXT_COLS].fillna(0.0)

assert len(df_ext) == len(df), "External merge duplicated team-seasons"
assert not df[BASE_FEATURES].isna().any().any(), "NaN in base features"
assert not df_ext[EXT_FEATURES].isna().any().any(), "NaN in external features"

print(f"Team-seasons: {len(df)} | clean years: {CLEAN_YEARS}")
print(f"Selected groups: {SELECTED_GROUPS}")
print(f"Base features ({len(BASE_FEATURES)}): {BASE_FEATURES}")
print(f"External features ({len(EXT_FEATURES)}): {EXT_FEATURES}")

Team-seasons: 732 | clean years: [2005, 2006, 2007, 2008, 2009]
Selected groups: ['prev_points', 'shots_luck', 'experience']
Base features (7): ['pts_pct_z_lag1', 'pts_pct_z_lag2', 'pts_pct_z_lag3', 'pts_pct_z_lag4', 'pts_pct_z_lag5', 'pts_pct_z_mean3', 'pts_pct_z_mean5']
External features (9): ['pts_pct_z_lag1', 'sh_pct_lag1_z', 'sv_pct_lag1_z', 'pdo_lag1_z', 'shot_diff_lag1_z', 'shot_diff_pct_lag1_z', 'pdo_mean3_z', 'shot_diff_pct_mean3_z', 'n_hist']


## 2. Models

All models output a score per team; the ranking is obtained by sorting the scores inside each
`(season, conference)`. Hyper-parameters are fixed to conservative values; nothing is tuned on the
test seasons.

The pairwise models learn from all within-conference pairs of the training seasons
(`P(A finishes ahead of B)` on feature differences), then aggregate the pairwise probabilities into
one score per team.

In [2]:
# ---- model factories -------------------------------------------------------
ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
boosting = lambda: HistGradientBoostingRegressor(
    max_depth=3, learning_rate=0.03, max_iter=250,
    min_samples_leaf=20, l2_regularization=1.0, random_state=RS,
)

MODELS = {
    "Ridge + external": {
        "family": "External",
        "features": EXT_FEATURES,
        "data": "external",
        "factory": ridge,
    },
    "Gradient boosting + external": {
        "family": "External",
        "features": EXT_FEATURES,
        "data": "external",
        "factory": boosting,
    },
    "Ensemble + external": {
        "family": "External",
        "features": EXT_FEATURES,
        "data": "external",
        "factory": lambda: VotingRegressor([
            ("ridge", ridge()), ("gbm", boosting()),
        ]),
    },
    "Pairwise logistic": {
        "family": "Pairwise",
        "features": BASE_FEATURES,
        "data": "base",
        "factory": lambda: make_pipeline(
            StandardScaler(with_mean=False),
            LogisticRegression(C=0.1, fit_intercept=False, max_iter=2000),
        ),
    },
    "Pairwise boosting": {
        "family": "Pairwise",
        "features": BASE_FEATURES,
        "data": "base",
        "factory": lambda: HistGradientBoostingClassifier(
            max_depth=3, learning_rate=0.05, max_iter=150,
            min_samples_leaf=50, l2_regularization=1.0, random_state=RS,
        ),
    },
}

NAIVE = {
    "Naive: random": "Random scores, fixed seed (no skill, floor).",
    "Naive: previous available ranking": (
        "Last available season's points percentage, re-ranked in the current conference; "
        "league average when the team has no history."
    ),
}
FAMILY = {name: "Naive" for name in NAIVE}
FAMILY.update({name: cfg["family"] for name, cfg in MODELS.items()})


# ---- pairwise ranker -------------------------------------------------------
def pair_differences(group, features):
    '''Feature differences (better team - worse team) for all unordered pairs.'''
    X = group[features].to_numpy(float)
    rank = group["conf_rank"].to_numpy()
    i, j = np.triu_indices(len(group), k=1)
    better_first = rank[i] < rank[j]
    better = np.where(better_first, i, j)
    worse = np.where(better_first, j, i)
    return X[better] - X[worse]


class PairwiseRanker:
    def __init__(self, make_classifier):
        self.make_classifier = make_classifier

    def fit(self, train, features):
        self.features = features
        diffs = [
            pair_differences(g, features)
            for (_, conf), g in train.groupby(["year", "confID"])
            if conf != "NONE" and len(g) >= 2
        ]
        D = np.vstack(diffs)
        X = np.vstack([D, -D])
        y = np.r_[np.ones(len(D)), np.zeros(len(D))]
        self.clf = self.make_classifier().fit(X, y)
        return self

    def score(self, test):
        scores = pd.Series(0.0, index=test.index)
        for _, g in test.groupby(["year", "confID"]):
            n = len(g)
            if n < 2:
                continue
            X = g[self.features].to_numpy(float)
            i, j = np.where(~np.eye(n, dtype=bool))
            d = X[i] - X[j]
            p = (self.clf.predict_proba(d)[:, 1]
                 + 1 - self.clf.predict_proba(-d)[:, 1]) / 2
            scores.loc[g.index] = np.bincount(i, weights=p, minlength=n) / (n - 1)
        return scores


def walk_forward_pairwise(data, make_classifier, features, eval_years):
    outputs = []
    for year in eval_years:
        train = data[(data["year"] < year) & (data["n_hist"] >= 1)]
        test = data[data["year"] == year]
        if test.empty:
            continue
        ranker = PairwiseRanker(make_classifier).fit(train, features)
        out = test[["year", "tmID", "confID"]].copy()
        out["score"] = ranker.score(test)
        outputs.append(out)
    return pd.concat(outputs, ignore_index=True)


print(f"{len(NAIVE)} naive baselines + {len(MODELS)} models")
for name, cfg in MODELS.items():
    print(f"  {cfg['family']:9s} {name:30s} {len(cfg['features'])} features ({cfg['data']})")

2 naive baselines + 5 models
  External  Ridge + external               9 features (external)
  External  Gradient boosting + external   9 features (external)
  External  Ensemble + external            9 features (external)
  Pairwise  Pairwise logistic              7 features (base)
  Pairwise  Pairwise boosting              7 features (base)


## 3. Walk-forward predictions

For each split, every model is retrained for every test season. The random baseline is redrawn with
the same seed in both splits. Predicted ranks, errors and the true ranks are stored together, so the
analysis notebook does not need the original data.

In [3]:
def predict_split(split, years):
    preds = {}

    rng = np.random.default_rng(RS)
    random_pred = df.loc[df["year"].isin(years), ["year", "tmID", "confID"]].copy()
    random_pred["score"] = rng.normal(size=len(random_pred))
    preds["Naive: random"] = random_pred

    preds["Naive: previous available ranking"] = ru.baseline_scores(
        df, "pts_pct_z_lag1", years
    )

    for name, cfg in MODELS.items():
        data = df_ext if cfg["data"] == "external" else df
        if cfg["family"] == "Pairwise":
            pred = walk_forward_pairwise(data, cfg["factory"], cfg["features"], years)
        else:
            pred = ru.walk_forward(data, cfg["factory"], cfg["features"], years)
        preds[name] = pred

    frames = []
    for name, pred in preds.items():
        ranked = ru.rank_predictions(pred, df)
        ranked["split"] = split
        ranked["model"] = name
        ranked["family"] = FAMILY[name]
        frames.append(ranked[[
            "split", "model", "family", "year", "tmID", "confID",
            "score", "pred_rank", "conf_rank", "conf_size",
            "target", "err", "abs_err",
        ]])
    return pd.concat(frames, ignore_index=True)


predictions = pd.concat(
    [predict_split(split, years) for split, years in SPLITS.items()],
    ignore_index=True,
)

print(f"Prediction rows: {len(predictions)}")
print(predictions.groupby(["split", "model"]).size().unstack(fill_value=0).T)

Prediction rows: 3871
split                              backtest  clean
model                                             
Ensemble + external                     403    150
Gradient boosting + external            403    150
Naive: previous available ranking       403    150
Naive: random                           403    150
Pairwise boosting                       403    150
Pairwise logistic                       403    150
Ridge + external                        403    150


## 4. Metrics and saved results

`metrics_by_season.csv` keeps one row per ranking, which is the right unit for the paired bootstrap
in the analysis notebook. `summary.csv` is the leaderboard.

In [4]:
def pair_accuracy(r):
    taus = [
        kendalltau(g["pred_rank"], g["conf_rank"])[0]
        for _, g in r.groupby(["year", "confID"])
    ]
    return (1 + np.mean(taus)) / 2


season_rows = []
for (split, model), r in predictions.groupby(["split", "model"]):
    table = ru.season_metrics(r)
    table["split"] = split
    table["model"] = model
    table["family"] = FAMILY[model]
    season_rows.append(table)
metrics_by_season = pd.concat(season_rows, ignore_index=True)

summary = (
    metrics_by_season
    .groupby(["split", "model", "family"])
    .agg(
        mae=("mae", "mean"),
        rmse=("rmse", "mean"),
        spearman=("spearman", "mean"),
        top_overlap=("top_overlap", "mean"),
        within_2=("within_2", "mean"),
        n_rankings=("mae", "size"),
    )
    .reset_index()
)
summary["pair_accuracy"] = [
    pair_accuracy(predictions[(predictions["split"] == row.split) &
                              (predictions["model"] == row.model)])
    for row in summary.itertuples()
]
summary = summary.sort_values(["split", "mae", "spearman"], ascending=[True, True, False])

# ---- save ------------------------------------------------------------------
ru.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
predictions.to_csv(ru.RESULTS_DIR / "predictions.csv", index=False, float_format="%.6f")
metrics_by_season.to_csv(ru.RESULTS_DIR / "metrics_by_season.csv", index=False, float_format="%.6f")
summary.to_csv(ru.RESULTS_DIR / "summary.csv", index=False, float_format="%.6f")

run_info = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "splits": {split: list(years) for split, years in SPLITS.items()},
    "selected_groups": SELECTED_GROUPS,
    "base_features": BASE_FEATURES,
    "external_features": EXT_FEATURES,
    "naive": NAIVE,
    "models": {
        name: {"family": cfg["family"], "data": cfg["data"],
               "n_features": len(cfg["features"]), "features": cfg["features"]}
        for name, cfg in MODELS.items()
    },
}
(ru.RESULTS_DIR / "run_info.json").write_text(
    json.dumps(run_info, indent=2), encoding="utf-8"
)

clean = summary[summary["split"] == "clean"].copy()
best = clean.iloc[0]
print(f"Saved results to {ru.RESULTS_DIR}")
print(f"Best clean model: {best['model']} | MAE {best['mae']:.3f} | "
      f"Spearman {best['spearman']:.3f} | top-8 {best['top_overlap']:.0%}")
display(
    clean[["family", "model", "mae", "rmse", "spearman", "top_overlap",
           "within_2", "pair_accuracy"]]
    .set_index(["family", "model"])
    .style.format({
        "mae": "{:.3f}", "rmse": "{:.3f}", "spearman": "{:.3f}",
        "top_overlap": "{:.1%}", "within_2": "{:.1%}", "pair_accuracy": "{:.1%}",
    })
)

Saved results to C:\Users\barcl\Work\Porto\AC\AC_hockey_predictions\ranking\results
Best clean model: Naive: previous available ranking | MAE 3.387 | Spearman 0.486 | top-8 64%


## 5. What to do next

The result files are the input of `ranking_results_analysis.ipynb`. Re-run this notebook only if
the data, the selected features or the model list change; the analysis notebook is cheap and can be
re-run on its own.